In [26]:
model_path = '/root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-1___5B-Instruct'

train_data_path = '../../dataset/prep/3/train.json'
test_file_path = '../../dataset/prep/3/test.json'
res_path = '../../dataset/res/tmp_res3.json'
save_path = "./output/save_model/"

lora_path = save_path+'checkpoint-80' 


# 导入环境

In [2]:
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, GenerationConfig

In [3]:
# 将JSON文件转换为CSV文件
df = pd.read_json(train_data_path)
ds = Dataset.from_pandas(df)

In [4]:
ds[:1]['input']

["\n[基本信息]:患者性别为男,职业为退休,年龄为62,婚姻为已婚,发病节气在大寒。\n[主诉]:心慌乏力7天。\n[症状]:阵发性心慌、乏力，无胸闷胸痛，无头晕头痛，纳眠可，二便调。\n[中医望闻切诊]:表情自然，面色红润，形体正常，动静姿态，语气清，气息平，无异常气味，舌暗红、舌苔白，脉沉数。\n[病史]:现病史，患者于7天前因无明显诱因出现阵发性心慌症状，乏力明显，为明确进一步中西医结合诊断治疗，遂入住我病区，入院症见，阵发性心慌、乏力，无胸闷胸痛，无头晕头痛，纳眠可，二便调，既往史，既往否认糖尿病等慢性疾病病史，否认肝炎、否认结核等传染病史，预防接种史不详，否认手术史、否认重大外伤史，否认输血史，否认药物过敏史、否认其他接触物过敏史，个人史，生于久居本地，无疫水、疫源接触史，无嗜酒史，无吸烟史，无放射线物质接触史，否认麻醉毒品等嗜好，否认冶游史，否认食物过敏史，否认传染病史，婚育史，适龄婚育，家族史，兄弟姐妹数人，均体健，否认家族性遗传病史。\n[体格检查]:生命体征体温：36.5℃  脉搏：150次/分  呼吸：19次/分  血压：124/83mmHg  \nPadua评分：1分    卒中风险评估：高危\n一般情况：患者中老年男性，发育正常，营养良好，神志清楚，自主步入病房，查体合作，皮肤黏膜：全身皮肤及粘膜无黄染，未见皮下出血,淋巴结浅表淋巴结未及肿大。标题定位符头颅五官无畸形，眼睑无水肿，巩膜无黄染，双侧瞳孔等大等圆，对光反射灵敏，外耳道无异常分泌物，鼻外观无畸形，口唇红润，伸舌居中，双侧扁桃体正常，表面未见脓性分泌物，标题定位符颈软，无抵抗感，双侧颈静脉正常，气管居中，甲状腺未及肿大，未闻及血管杂音。标题定位符胸廓正常，双肺呼吸音清晰，未闻及干、湿啰音，未闻及胸膜摩擦音。心脏心界不大，心率160次/分，心律不齐，心音强弱不等，各瓣膜听诊区未闻及杂音，未闻及心包摩擦音。脉搏短绌，无水冲脉、枪击音、毛细血管搏动征。腹部腹部平坦，无腹壁静脉显露，无胃肠型和蠕动波，腹部柔软，无压痛、反跳痛，肝脏未触及，脾脏未触及，未触及腹部包块，麦氏点无压痛及反跳痛，Murphy's征－，肾脏未触及，肝浊音界正常，无明显肾区叩击痛，肝脾区无明显叩击痛，腹部叩诊鼓音，移动性浊音-，肠鸣音正常，4次/分，无过水声，直肠肛门、生殖器肛门及外生殖器未查，脊柱四肢脊柱生理弯曲存

# 处理数据集

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, trust_remote_code=True)
tokenizer

Qwen2Tokenizer(name_or_path='/root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-1___5B-Instruct', vocab_size=151643, model_max_length=131072, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=Fals

In [6]:
def process_func(example):
    MAX_LENGTH = 4096    # Llama分词器会将一个中文字切分为多个token，因此需要放开一些最大长度，保证数据的完整性
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer(f"<|im_start|>system\n现在进行一个中医处方生成任务<|im_end|>\n<|im_start|>user\n{example['instruction'] + example['input']}<|im_end|>\n<|im_start|>assistant\n", add_special_tokens=False)  # add_special_tokens 不在开头加 special_tokens
    response = tokenizer(f"{example['output']}", add_special_tokens=False)
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]  # 因为eos token咱们也是要关注的所以 补充为1
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]  
    if len(input_ids) > MAX_LENGTH:  # 做一个截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [7]:
tokenized_id = ds.map(process_func, remove_columns=ds.column_names)
tokenized_id

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 800
})

In [8]:
tokenizer.decode(tokenized_id[0]['input_ids'])

"<|im_start|>system\n现在进行一个中医处方生成任务<|im_end|>\n<|im_start|>user\n\n任务：根据患者[基本信息],[主诉],[症状],[中医望闻切诊],[病史],[体格检查],[辅助检查]等信息,在[草药]中为患者推荐需要使用的[推荐草药]。\n# 要求1: 草药局限在下方列表中。\n[草药]: '冬瓜皮', '沉香', '茜草炭', '浮小麦', '炙甘草', '炒白扁豆', '砂仁', '合欢花', '北刘寄奴', '炒六神曲', '炒决明子', '益母草', '酒苁蓉', '炒僵蚕', '稀莶草', '秦艽', '黄酒', '瞿麦', ' 白鲜皮', '熟地黄', '扁蓄', '诃子肉', '煅牡蛎', '鸡血藤', '党参', '瓜蒌', '莲子', '酒五味子', '金钱草', '法半夏', '北败酱草', '花椒', '吴茱萸(粉)', '桑白皮', '茯神', '桂枝', '降香', '制远志', '琥珀', '佛手', '麦芽', '水红花子', '金银花', '马鞭草', '半枝莲', '炮姜', '生酸枣仁', '盐补骨脂', '炒瓜蒌子', '珍珠母', '乌药', '茵陈', '地肤子', '酸枣仁', '槟榔', '大青叶', '人参片', '麸煨肉豆蔻', '蛤蚧', '路路通', '蝉蜕', '马勃', '香橼', '络石藤', '狗脊', '蜈蚣', '制川乌', '白扁豆花', '麻黄', '射干', '厚朴', '蜂蜜', '柏子仁', '炒谷芽', '蜜百合', ' 石菖蒲', '白薇', '续断', '炒川楝子', '黄连片', '绵萆薢', '鹿角胶', '翻白草', '羚羊角粉', '天麻', '山慈菇', '菊花', '炒芥子', '墨旱莲', '蜜枇杷叶', '川芎', '酒大黄', '焦山楂', '红曲', '山药', '牡蛎', '海藻', '夏枯草', '白前', '白芍', '茯苓皮', '煅自然铜', '附片 ', '土茯苓', '制何首乌', '炒莱菔子', '黄芩', '蒲黄', '紫石英', '透骨草', '绞股蓝', '泽泻', '甘松', ' 炒酸枣仁', '儿茶', '马齿苋', '太子参', '薏苡仁', '萹蓄', '青蒿', '苏木', '桑叶

In [9]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_id[1]["labels"])))

"'干姜', '黄柏', '酒大黄', '肉桂', '附片 ', '山药', '菟丝子', '淫羊藿', '牡蛎', '柴胡', '黄芩', '焦栀子', '牡丹皮', '茯苓', '龙骨', '桂枝', '法半夏'<|endoftext|>"

# 创建模型

In [10]:
import torch

model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto",torch_dtype=torch.bfloat16)
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qw

In [11]:
model.enable_input_require_grads() # 开启梯度检查点时，要执行该方法

In [12]:
model.dtype

torch.bfloat16

In [13]:
!nvidia-smi

Sat Mar  8 13:01:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100S-PCIE-32GB          On  |   00000000:00:0A.0 Off |                  Off |
| N/A   43C    P0             60W /  250W |    4004MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# lora 

In [14]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=False, # 训练模式
    r=8, # Lora 秩
    lora_alpha=32, # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1# Dropout 比例
)
config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=8, target_modules={'k_proj', 'up_proj', 'v_proj', 'q_proj', 'o_proj', 'gate_proj', 'down_proj'}, lora_alpha=32, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None)

In [15]:
model = get_peft_model(model, config)
config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='/root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-1___5B-Instruct', revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=8, target_modules={'k_proj', 'up_proj', 'v_proj', 'q_proj', 'o_proj', 'gate_proj', 'down_proj'}, lora_alpha=32, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None)

In [16]:
model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945074657965335


# 配置训练参数

In [17]:
args = TrainingArguments(
    output_dir=save_path,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    logging_steps=1,
    num_train_epochs=1,
    save_steps=10, 
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True
)


In [18]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_id,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

In [19]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/root/miniconda3/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/miniconda3/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss
1,2.300900
2,1.689900
3,1.680000
4,1.680800
5,1.430700
6,1.344700
7,1.418200
8,1.366300
9,1.389100
10,1.330400


/root/miniconda3/lib/python3.10/site-packages/peft/utils/save_and_load.py:154: UserWarning: Could not find a config file in /root/autodl-tmp/self-llm/model/Qwen/Qwen2___5-1___5B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
/root/miniconda3/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/miniconda3/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.auto

In [23]:
!nvidia-smi

Sat Mar  8 13:26:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100S-PCIE-32GB          On  |   00000000:00:0A.0 Off |                  Off |
| N/A   40C    P0             39W /  250W |   17930MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 合并加载模型

In [24]:
import json

with open(test_file_path, 'r', encoding='utf-8') as file:
    # 读取并解析 JSON 数据
    test_data = json.load(file)
    # 打印读取的数据
    print(test_data[0])

{'ID': '1405', 'instruction': "\n任务：根据患者[基本信息],[主诉],[症状],[中医望闻切诊],[病史],[体格检查],[辅助检查]等信息,在[草药]中为患者推荐需要使用的[推荐草药]。\n# 要求1: 草药局限在下方列表中。\n[草药]: '冬瓜皮', '沉香', '茜草炭', '浮小麦', '炙甘草', '炒白扁豆', '砂仁', '合欢花', '北刘寄奴', '炒六神曲', '炒决明子', '益母草', '酒苁蓉', '炒僵蚕', '稀莶草', '秦艽', '黄酒', '瞿麦', ' 白鲜皮', '熟地黄', '扁蓄', '诃子肉', '煅牡蛎', '鸡血藤', '党参', '瓜蒌', '莲子', '酒五味子', '金钱草', '法半夏', '北败酱草', '花椒', '吴茱萸(粉)', '桑白皮', '茯神', '桂枝', '降香', '制远志', '琥珀', '佛手', '麦芽', '水红花子', '金银花', '马鞭草', '半枝莲', '炮姜', '生酸枣仁', '盐补骨脂', '炒瓜蒌子', '珍珠母', '乌药', '茵陈', '地肤子', '酸枣仁', '槟榔', '大青叶', '人参片', '麸煨肉豆蔻', '蛤蚧', '路路通', '蝉蜕', '马勃', '香橼', '络石藤', '狗脊', '蜈蚣', '制川乌', '白扁豆花', '麻黄', '射干', '厚朴', '蜂蜜', '柏子仁', '炒谷芽', '蜜百合', ' 石菖蒲', '白薇', '续断', '炒川楝子', '黄连片', '绵萆薢', '鹿角胶', '翻白草', '羚羊角粉', '天麻', '山慈菇', '菊花', '炒芥子', '墨旱莲', '蜜枇杷叶', '川芎', '酒大黄', '焦山楂', '红曲', '山药', '牡蛎', '海藻', '夏枯草', '白前', '白芍', '茯苓皮', '煅自然铜', '附片 ', '土茯苓', '制何首乌', '炒莱菔子', '黄芩', '蒲黄', '紫石英', '透骨草', '绞股蓝', '泽泻', '甘松', ' 炒酸枣仁', '儿茶', '马齿苋', '太子参', '薏苡仁', '萹蓄', '青蒿', '苏木', '桑叶', '连翘', '穿山龙', '忍冬藤', '苦参', '炒茺蔚子

In [27]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import PeftModel

# 加载tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# 加载模型
# model = AutoModelForCausalLM.from_pretrained(mode_path, , device_map="cuda:0",torch_dtype=torch.bfloat16, trust_remote_code=True).eval()

# 加载lora权重
model = PeftModel.from_pretrained(model, model_id=lora_path)



In [28]:
from tqdm import tqdm

# 假设 test_data、tokenizer 和 model 已经定义
for d in tqdm(test_data, desc="Processing test data"):
    prompt = d['input']
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": d['instruction']},{"role": "user", "content": prompt}],
                                           add_generation_prompt=True,
                                           tokenize=True,
                                           return_tensors="pt",
                                           return_dict=True
                                           ).to('cuda')
    
    
    gen_kwargs = {"max_length": 4096, "do_sample": True, "top_k": 1}
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        outputs = outputs[:, inputs['input_ids'].shape[1]:]
        d['output'] = tokenizer.decode(outputs[0], skip_special_tokens=True)

Processing test data: 100%|██████████| 200/200 [11:21<00:00,  3.41s/it]


In [ ]:
# 方法一：使用 with 语句自动管理文件的打开和关闭

try:
    with open(res_path, 'w', encoding='utf-8') as file:
        # 将 list 转换为 JSON 格式并写入文件
        json.dump(test_data, file, ensure_ascii=False, indent=4)
    print(f"数据已成功保存到 {file_path}")
except Exception as e:
    print(f"保存文件时出现错误: {e}")

In [ ]:
!nvidia-smi

In [ ]:
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi